In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../datasets/adult.csv')


In [2]:
df.shape

(48842, 15)

In [3]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              48842 non-null  int64
 1   workclass        48842 non-null  str  
 2   fnlwgt           48842 non-null  int64
 3   education        48842 non-null  str  
 4   educational-num  48842 non-null  int64
 5   marital-status   48842 non-null  str  
 6   occupation       48842 non-null  str  
 7   relationship     48842 non-null  str  
 8   race             48842 non-null  str  
 9   gender           48842 non-null  str  
 10  capital-gain     48842 non-null  int64
 11  capital-loss     48842 non-null  int64
 12  hours-per-week   48842 non-null  int64
 13  native-country   48842 non-null  str  
 14  income           48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [5]:
print((df == '?').sum())

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


#### **here missing values have '?' as a placeholder**
#### so first lets convert it into **NAN**

In [6]:
df = df.replace('?', np.nan)

print(df.isnull().sum())

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
educational-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


In [8]:
# checking the missing value percentage

total = len(df)

missing = df.isnull().sum()
missing_pct = (missing/total) * 100

missing_df = pd.DataFrame({
    'missing_count' : missing,
    'missing_pct' : missing_pct
})

missing_df[missing_df['missing_count']  > 0]

,missing_count,missing_pct
workclass,2799,5.730724
occupation,2809,5.751198
native-country,857,1.754637


In [9]:
both_missing = df['workclass'].isnull() & df['occupation'].isnull()
print(both_missing.sum())

2799


In [10]:
nc_missing = df['native-country'].isnull()
print("native_country missing that also miss workclass:", (nc_missing & both_missing).sum())
print("native-country missing independently:", (nc_missing & ~both_missing).sum())

native_country missing that also miss workclass: 46
native-country missing independently: 811


In [11]:
# dropping NA values from workclass and occupation

df = df.dropna(subset=['workclass', 'occupation'])

print(df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])

(46033, 15)
native-country    811
dtype: int64


In [12]:
# filling native country missing values with "Unknown"

df['native-country'] = df['native-country'].fillna('Unknown')
print(df.isnull().sum().sum())

0


In [13]:
# dropping the redundant columns

df = df.drop(columns=['fnlwgt', 'educational-num'])

print(df.shape)
print(df.columns.tolist())

(46033, 13)
['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['income'])
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

print(X_train.shape, X_test.shape)

(36826, 12) (9207, 12)


In [15]:
X_train.dtypes

age               int64
workclass           str
education           str
marital-status      str
occupation          str
relationship        str
race                str
gender              str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
dtype: object

In [16]:
for col in X_train.select_dtypes(include='str').columns:
    print(f"{col}: {X_train[col].nunique()} unique values")

workclass: 7 unique values
education: 16 unique values
marital-status: 7 unique values
occupation: 14 unique values
relationship: 6 unique values
race: 5 unique values
gender: 2 unique values
native-country: 42 unique values


In [17]:
print(sorted(X_train['education'].unique()))

['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th', '9th', 'Assoc-acdm', 'Assoc-voc', 'Bachelors', 'Doctorate', 'HS-grad', 'Masters', 'Preschool', 'Prof-school', 'Some-college']


In [18]:
education_order = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th',
    '10th', '11th', '12th', 'HS-grad', 'Some-college',
    'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
    'Prof-school', 'Doctorate'
]

In [19]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

ordinal_cols = ['gender', 'education']
ohe_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race']
numerical_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week']

preprocessor = ColumnTransformer(transformers=[
    ('ordinal', OrdinalEncoder(categories=[['Male', 'Female'], education_order]), ordinal_cols),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols),
    ('scaler', StandardScaler(), numerical_cols)
], remainder='drop')

In [20]:
from sklearn.preprocessing import TargetEncoder
import numpy as np

# step 1 - fit and transform preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# step 2 - fit and transform target encoder
te = TargetEncoder(target_type='binary')
X_train_country = te.fit_transform(X_train[['native-country']], y_train)
X_test_country = te.transform(X_test[['native-country']])

# step 3 - combine
X_train_final = np.hstack([X_train_processed,X_train_country])
X_test_final = np.hstack([X_test_processed, X_test_country])

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

X_train_final shape: (36826, 46)
X_test_final shape: (9207, 46)


In [22]:
ohe_feature_names = preprocessor.named_transformers_['ohe'].get_feature_names_out(ohe_cols)

feature_names = list(ordinal_cols) + list(ohe_feature_names) + list(numerical_cols) + ['native-country']

df_final = pd.DataFrame(X_train_final, columns=feature_names)
df_final.head()

,gender,education,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,marital-status_Divorced,...,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,age,capital-gain,capital-loss,hours-per-week,native-country
0,0.0,10.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,-0.875809,-0.146791,-0.221334,-0.078192,0.251588
1,0.0,6.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,-1.556328,-0.146791,-0.221334,-1.747768,0.251986
2,1.0,12.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,-0.497743,-0.146791,-0.221334,-1.413853,0.251388
3,0.0,12.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.258390,-0.146791,-0.221334,2.008776,0.251388
4,0.0,12.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,-0.951422,-0.146791,-0.221334,-2.165161,0.251388


In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(le.classes_)
print(y_train_encoded[:5])

['<=50K' '>50K']
[0 0 1 0 0]


In [24]:
unique, counts = np.unique(y_train_encoded, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c} ({c/len(y_train_encoded)*100:.1f}%)")

Class 0: 27749 (75.4%)
Class 1: 9077 (24.6%)


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_final, y_train_encoded)

y_pred = model.predict(X_test_final)
print(classification_report(y_test_encoded, y_pred, target_names=['<=50K', '>50K']))

              precision    recall  f1-score   support

       <=50K       0.93      0.80      0.86      6862
        >50K       0.58      0.84      0.69      2345

    accuracy                           0.81      9207
   macro avg       0.76      0.82      0.77      9207
weighted avg       0.85      0.81      0.82      9207



In [26]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test_encoded, y_pred))

0.8067774519387423
